In [1]:
import os, json, time, random, sqlite3
from datetime import datetime, timedelta
import pandas as pd
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic()
MODEL = "claude-haiku-5-5"
print("Ready! Key found:", os.getenv("ANTHROPIC_API_KEY", "")[:10] + "...")

Ready! Key found: sk-ant-usr...


In [2]:
random.seed(42)
if os.path.exists("quickcommerce.db"):
    os.remove("quickcommerce.db")
con = sqlite3.connect("quickcommerce.db")
cur = con.cursor()

cur.execute("CREATE TABLE stores (store_id INTEGER PRIMARY KEY, city TEXT, zone TEXT, opened_date TEXT)")
cur.execute("CREATE TABLE riders (rider_id INTEGER PRIMARY KEY, city TEXT, vehicle_type TEXT, join_date TEXT)")
cur.execute("""CREATE TABLE orders (order_id INTEGER PRIMARY KEY, store_id INTEGER, rider_id INTEGER,
               customer_id INTEGER, order_time TEXT, promised_minutes INTEGER, delivered_time TEXT,
               status TEXT, amount REAL, items_count INTEGER)""")

CITIES = {"Bengaluru": ["North", "South", "East", "West"], "Delhi": ["North", "South", "East", "West"],
          "Mumbai": ["North", "South", "West"], "Hyderabad": ["North", "South"], "Pune": ["East", "West"]}
stores, store_id = [], 1
for city, zones in CITIES.items():
    for zone in zones:
        for _ in range(3):
            opened = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 600))
            stores.append((store_id, city, zone, opened.strftime("%Y-%m-%d")))
            store_id += 1
cur.executemany("INSERT INTO stores VALUES (?,?,?,?)", stores)

riders = []
for rider_id in range(1, 301):
    city = random.choice(list(CITIES))
    vehicle = "EV" if random.random() < (0.55 if city == "Bengaluru" else 0.35) else "Petrol"
    joined = datetime(2025, 1, 1) + timedelta(days=random.randint(0, 600))
    riders.append((rider_id, city, vehicle, joined.strftime("%Y-%m-%d")))
cur.executemany("INSERT INTO riders VALUES (?,?,?,?)", riders)

riders_by_city = {}
for r in riders:
    riders_by_city.setdefault(r[1], []).append(r)

orders, start = [], datetime(2026, 9, 1)
for order_id in range(1, 6001):
    s = random.choice(stores)
    r = random.choice(riders_by_city[s[1]])
    t = start + timedelta(days=random.randint(0, 60), hours=random.randint(7, 23), minutes=random.randint(0, 59))
    promised = random.choice([10, 15])
    roll = random.random()
    status = "cancelled" if roll < 0.04 else ("mdnd" if roll < 0.055 else "delivered")
    late_bias = 6 if r[2] == "Petrol" else 3
    minutes = max(5, int(random.gauss(promised - 3, late_bias)))
    delivered = (t + timedelta(minutes=minutes)).strftime("%Y-%m-%d %H:%M") if status != "cancelled" else None
    amount = round(random.uniform(99, 1500), 2)
    orders.append((order_id, s[0], r[0], random.randint(1, 2000), t.strftime("%Y-%m-%d %H:%M"),
                   promised, delivered, status, amount, random.randint(1, 12)))
cur.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?,?,?,?,?)", orders)
con.commit()
con.close()
print("Built quickcommerce.db with", len(stores), "stores,", len(riders), "riders,", len(orders), "orders")

Built quickcommerce.db with 45 stores, 300 riders, 6000 orders


In [3]:
def run_sql(sql):
    con = sqlite3.connect("file:quickcommerce.db?mode=ro", uri=True)  # ro = read-only, cannot change data
    try:
        return pd.read_sql_query(sql, con)
    finally:
        con.close()

run_sql("SELECT * FROM orders LIMIT 5")

,order_id,store_id,rider_id,customer_id,order_time,promised_minutes,delivered_time,status,amount,items_count
0,1,32,292,936,2026-09-03 11:32,15,2026-09-03 11:48,delivered,835.86,1
1,2,10,140,1277,2026-10-26 11:04,15,2026-10-26 11:13,delivered,470.30,12
2,3,26,211,1109,2026-09-06 17:54,15,2026-09-06 18:01,delivered,782.61,1
3,4,40,32,1298,2026-09-16 16:14,10,2026-09-16 16:19,delivered,236.92,12
4,5,7,148,299,2026-09-11 16:57,10,2026-09-11 17:05,mdnd,624.15,4


In [4]:
LATE = "(julianday(delivered_time) - julianday(order_time)) * 24 * 60 > promised_minutes"
TESTS = [
    # --- easy ---
    ("How many orders are there in total?", "SELECT COUNT(*) FROM orders"),
    ("How many stores are in Bengaluru?", "SELECT COUNT(*) FROM stores WHERE city = 'Bengaluru'"),
    ("How many riders use an EV?", "SELECT COUNT(*) FROM riders WHERE vehicle_type = 'EV'"),
    ("What is the average order amount?", "SELECT ROUND(AVG(amount), 2) FROM orders"),
    ("How many orders were cancelled?", "SELECT COUNT(*) FROM orders WHERE status = 'cancelled'"),
    ("What is the highest single order amount?", "SELECT MAX(amount) FROM orders"),
    ("How many orders were placed in October 2026?", "SELECT COUNT(*) FROM orders WHERE order_time >= '2026-10-01' AND order_time < '2026-11-01'"),
    ("How many different customers have placed an order?", "SELECT COUNT(DISTINCT customer_id) FROM orders"),
    ("What is the total delivered order value (GMV)?", "SELECT ROUND(SUM(amount), 2) FROM orders WHERE status = 'delivered'"),
    ("How many riders joined in 2026?", "SELECT COUNT(*) FROM riders WHERE join_date >= '2026-01-01'"),
    # --- medium ---
    ("How many orders did each city get? Show city and order count.",
     "SELECT s.city, COUNT(*) FROM orders o JOIN stores s ON o.store_id = s.store_id GROUP BY s.city"),
    ("What percentage of riders in each city use EVs?",
     "SELECT city, ROUND(100.0 * SUM(vehicle_type = 'EV') / COUNT(*), 1) FROM riders GROUP BY city"),
    ("What is the MDND rate (as a %) overall?",
     "SELECT ROUND(100.0 * SUM(status = 'mdnd') / COUNT(*), 2) FROM orders"),
    ("Which 5 stores had the most orders? Show store_id and order count. If tied, the lower store_id comes first.",
     "SELECT store_id, COUNT(*) AS n FROM orders GROUP BY store_id ORDER BY n DESC, store_id LIMIT 5"),
    ("What is the average order amount by zone in Mumbai?",
     "SELECT s.zone, ROUND(AVG(o.amount), 2) FROM orders o JOIN stores s ON o.store_id = s.store_id WHERE s.city = 'Mumbai' GROUP BY s.zone"),
    ("How many orders did EV riders deliver compared to Petrol riders? Count delivered orders by vehicle type.",
     "SELECT r.vehicle_type, COUNT(*) FROM orders o JOIN riders r ON o.rider_id = r.rider_id WHERE o.status = 'delivered' GROUP BY r.vehicle_type"),
    ("Which city has the highest cancellation rate? Show the city and the rate as a %.",
     "SELECT s.city, ROUND(100.0 * SUM(o.status = 'cancelled') / COUNT(*), 2) AS rate FROM orders o JOIN stores s ON o.store_id = s.store_id GROUP BY s.city ORDER BY rate DESC LIMIT 1"),
    ("How many orders were placed on each day of the week? Use SQLite day numbers 0 (Sunday) to 6.",
     "SELECT strftime('%w', order_time) AS dow, COUNT(*) FROM orders GROUP BY dow"),
    ("What is the average number of items per order for orders above Rs 1000?",
     "SELECT ROUND(AVG(items_count), 2) FROM orders WHERE amount > 1000"),
    ("Which 3 riders delivered the most orders? Show rider_id and delivered count. If tied, the lower rider_id comes first.",
     "SELECT rider_id, COUNT(*) AS n FROM orders WHERE status = 'delivered' GROUP BY rider_id ORDER BY n DESC, rider_id LIMIT 3"),
    # --- hard ---
    ("What % of delivered orders were late? Late means delivery took longer than promised_minutes.",
     f"SELECT ROUND(100.0 * SUM({LATE}) / COUNT(*), 2) FROM orders WHERE status = 'delivered'"),
    ("What is the late delivery % for EV riders vs Petrol riders?",
     "SELECT r.vehicle_type, ROUND(100.0 * SUM((julianday(o.delivered_time) - julianday(o.order_time)) * 24 * 60 > o.promised_minutes) / COUNT(*), 2) FROM orders o JOIN riders r ON o.rider_id = r.rider_id WHERE o.status = 'delivered' GROUP BY r.vehicle_type"),
    ("Which 5 stores have the highest late delivery %? Show store_id and the %. If tied, the lower store_id comes first.",
     f"SELECT store_id, ROUND(100.0 * SUM({LATE}) / COUNT(*), 2) AS late_pct FROM orders WHERE status = 'delivered' GROUP BY store_id ORDER BY late_pct DESC, store_id LIMIT 5"),
    ("What is the average delivery time in minutes for delivered orders, by city?",
     "SELECT s.city, ROUND(AVG((julianday(o.delivered_time) - julianday(o.order_time)) * 24 * 60), 2) FROM orders o JOIN stores s ON o.store_id = s.store_id WHERE o.status = 'delivered' GROUP BY s.city"),
    ("How many orders were placed each week? Use strftime('%W') week numbers.",
     "SELECT strftime('%W', order_time) AS wk, COUNT(*) FROM orders GROUP BY wk"),
    ("For each city, which store had the most orders? Show city, store_id and order count.",
     "SELECT city, store_id, n FROM (SELECT s.city, o.store_id, COUNT(*) AS n, RANK() OVER (PARTITION BY s.city ORDER BY COUNT(*) DESC) AS rk FROM orders o JOIN stores s ON o.store_id = s.store_id GROUP BY s.city, o.store_id) WHERE rk = 1"),
    ("How many customers placed more than 4 orders?",
     "SELECT COUNT(*) FROM (SELECT customer_id FROM orders GROUP BY customer_id HAVING COUNT(*) > 4)"),
    ("What is the daily order count in Delhi for the first 7 days of October 2026?",
     "SELECT DATE(o.order_time) AS d, COUNT(*) FROM orders o JOIN stores s ON o.store_id = s.store_id WHERE s.city = 'Delhi' AND o.order_time >= '2026-10-01' AND o.order_time < '2026-10-08' GROUP BY d"),
    ("What share of total GMV (delivered orders only) does each city contribute, as a %?",
     "SELECT s.city, ROUND(100.0 * SUM(o.amount) / (SELECT SUM(amount) FROM orders WHERE status = 'delivered'), 2) FROM orders o JOIN stores s ON o.store_id = s.store_id WHERE o.status = 'delivered' GROUP BY s.city"),
    ("How many riders delivered fewer than 15 orders (including riders with none)?",
     "SELECT COUNT(*) FROM (SELECT r.rider_id FROM riders r LEFT JOIN orders o ON o.rider_id = r.rider_id AND o.status = 'delivered' GROUP BY r.rider_id HAVING COUNT(o.order_id) < 15)"),
]
print("Test questions:", len(TESTS))
run_sql(TESTS[21][1])

Test questions: 30


,vehicle_type,"ROUND(100.0 * SUM((julianday(o.delivered_time) - julianday(o.order_time)) * 24 * 60 > o.promised_minutes) / COUNT(*), 2)"
0,EV,13.50
1,Petrol,28.48


In [5]:
def same_answer(df_a, df_b):
    """True if two results contain the same values (ignores column names and row order)."""
    def clean(df):
        rows = []
        for row in df.itertuples(index=False):
            rows.append(tuple(round(v, 1) if isinstance(v, float) else v for v in row))
        return sorted(rows, key=str)
    return clean(df_a) == clean(df_b)

In [6]:
def ask_claude(system_prompt, question):
    resp = client.messages.create(
        model=MODEL,
        max_tokens=800,
        system=system_prompt,
        thinking={"type": "disabled"},
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": question}],
    )
    sql = "".join(b.text for b in resp.content if b.type == "text").strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()  # remove code fences if any
    return sql

In [7]:
PROMPT_A = """Write one SQLite query that answers the question. Reply with only the SQL, nothing else.

Tables:
stores(store_id, city, zone, opened_date)
riders(rider_id, city, vehicle_type, join_date)
orders(order_id, store_id, rider_id, customer_id, order_time, promised_minutes, delivered_time, status, amount, items_count)
"""

def score(prompt, retry=False, label=""):
    rows = []
    for n, (question, gold_sql) in enumerate(TESTS, start=1):
        gold = run_sql(gold_sql)
        sql = ask_claude(prompt, question)
        try:
            if not sql.lower().lstrip().startswith(("select", "with")):
                raise ValueError("Only SELECT queries are allowed")
            got = run_sql(sql)
        except Exception as e:
            if not retry:
                got = None
            else:  # send the error back to Claude once
                sql = ask_claude(prompt, f"{question}\n\nYour previous SQL:\n{sql}\nfailed with this error:\n{e}\nWrite a corrected query.")
                try:
                    got = run_sql(sql)
                except Exception:
                    got = None
        correct = got is not None and same_answer(got, gold)
        rows.append({"q": n, "question": question, "claude_sql": sql, "correct": correct})
        print(f"{label} Q{n}: {'RIGHT' if correct else 'wrong'}")
    return pd.DataFrame(rows)

res_a = score(PROMPT_A, retry=False, label="A")
print(f"\nVersion A score: {res_a['correct'].sum()}/30   <-- this is your [X]/30")

A Q1: RIGHT
A Q2: RIGHT
A Q3: RIGHT
A Q4: RIGHT
A Q5: RIGHT
A Q6: RIGHT
A Q7: RIGHT
A Q8: RIGHT
A Q9: RIGHT
A Q10: RIGHT
A Q11: RIGHT
A Q12: RIGHT
A Q13: wrong
A Q14: RIGHT
A Q15: RIGHT
A Q16: RIGHT
A Q17: RIGHT
A Q18: wrong
A Q19: RIGHT
A Q20: RIGHT
A Q21: RIGHT
A Q22: wrong
A Q23: wrong
A Q24: RIGHT
A Q25: RIGHT
A Q26: RIGHT
A Q27: RIGHT
A Q28: RIGHT
A Q29: RIGHT
A Q30: RIGHT

Version A score: 26/30   <-- this is your [X]/30


In [8]:
PROMPT_B = """You are a data analyst at an Indian quick-commerce company. Write one SQLite query that answers the question.
Reply with only the SQL, nothing else. Only SELECT queries are allowed.

Tables and columns:
stores: one row per dark store
  - store_id: store number
  - city: Bengaluru, Delhi, Mumbai, Hyderabad or Pune
  - zone: North, South, East or West part of the city
  - opened_date: 'YYYY-MM-DD'
riders: one row per delivery rider
  - rider_id, city
  - vehicle_type: exactly 'EV' or 'Petrol'
  - join_date: 'YYYY-MM-DD'
orders: one row per customer order
  - order_id, store_id (links to stores), rider_id (links to riders), customer_id
  - order_time and delivered_time: text 'YYYY-MM-DD HH:MM'; delivered_time is NULL for cancelled orders
  - promised_minutes: delivery time promised to the customer (10 or 15)
  - status: exactly 'delivered', 'cancelled' or 'mdnd' (marked delivered, not delivered)
  - amount: order value in rupees
  - items_count: number of items

Business rules:
- GMV = SUM(amount) of delivered orders only.
- Delivery time in minutes = (julianday(delivered_time) - julianday(order_time)) * 24 * 60
- An order is late if its delivery time in minutes > promised_minutes. Late % is measured over delivered orders only.
- A rate or share "as a %" = 100.0 * part / total, rounded to 2 decimals.
- City of an order = the city of its store.
- Return only the columns the question asks for. Round averages to 2 decimals.

Examples:
Q: How many stores are in Pune?
SQL: SELECT COUNT(*) FROM stores WHERE city = 'Pune'
Q: What is the GMV of Hyderabad?
SQL: SELECT ROUND(SUM(o.amount), 2) FROM orders o JOIN stores s ON o.store_id = s.store_id WHERE s.city = 'Hyderabad' AND o.status = 'delivered'
Q: What % of orders in Pune were cancelled?
SQL: SELECT ROUND(100.0 * SUM(o.status = 'cancelled') / COUNT(*), 2) FROM orders o JOIN stores s ON o.store_id = s.store_id WHERE s.city = 'Pune'
"""

res_b = score(PROMPT_B, retry=True, label="B")
print(f"\nVersion B score: {res_b['correct'].sum()}/30   <-- this is your [Y]/30")

B Q1: RIGHT
B Q2: RIGHT
B Q3: RIGHT
B Q4: RIGHT
B Q5: RIGHT
B Q6: RIGHT
B Q7: RIGHT
B Q8: RIGHT
B Q9: RIGHT
B Q10: RIGHT
B Q11: RIGHT
B Q12: RIGHT
B Q13: RIGHT
B Q14: RIGHT
B Q15: RIGHT
B Q16: RIGHT
B Q17: RIGHT
B Q18: wrong
B Q19: RIGHT
B Q20: RIGHT
B Q21: RIGHT
B Q22: RIGHT
B Q23: RIGHT
B Q24: RIGHT
B Q25: RIGHT
B Q26: RIGHT
B Q27: RIGHT
B Q28: RIGHT
B Q29: RIGHT
B Q30: RIGHT

Version B score: 29/30   <-- this is your [Y]/30


In [9]:
compare = pd.DataFrame({"question": res_a["question"], "A_correct": res_a["correct"], "B_correct": res_b["correct"]})
compare.to_csv("results_project2.csv", index=False)
print(f"Version A: {res_a['correct'].sum()}/30")
print(f"Version B: {res_b['correct'].sum()}/30")
print("\nQuestions B still gets wrong:")
for r in res_b[~res_b["correct"]].itertuples():
    print(f"\nQ{r.q}: {r.question}\nClaude wrote: {r.claude_sql}")

Version A: 26/30
Version B: 29/30

Questions B still gets wrong:

Q18: How many orders were placed on each day of the week? Use SQLite day numbers 0 (Sunday) to 6.
Claude wrote: SELECT CAST(strftime('%w', order_time) AS INTEGER) AS day_of_week, COUNT(*) AS order_count FROM orders GROUP BY day_of_week ORDER BY day_of_week
